In [14]:
import os
import sys
import pandas as pd
import numpy as np
import dotenv

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

dotenv.load_dotenv(dotenv.find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))

INPUT_FILEPATH_1 = os.path.join(RAW_DATA_PATH, "sales-analysis-redfin/data/best_treatment_dates_2026-07.csv")
INPUT_FILEPATH_2 = os.path.join(RAW_DATA_PATH, "sales-analysis-redfin/data/str_dates_cities_51_100.csv")

OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "raw_data/regs_unit_list.csv")


In [15]:
regs_df_1 = pd.read_csv(INPUT_FILEPATH_1)
regs_df_2 = pd.read_csv(INPUT_FILEPATH_2)

In [16]:
# Cleaning regs_df_1

regs_df_1_clean = regs_df_1[['city', 'state', 'best_passage', 'best_enforcement']].rename(
    columns={
        'best_passage': 'passage_date',
        'best_enforcement': 'effective_date'
    }
)
regs_df_1_clean['passage_date'] = pd.to_datetime(regs_df_1_clean['passage_date'])
regs_df_1_clean['effective_date'] = pd.to_datetime(regs_df_1_clean['effective_date'])


In [17]:
# Cleaning regs_df_2

regs_df_2_clean = regs_df_2[['city', 'state', 'passage_date', 'effective_date']]
regs_df_2_clean['passage_date'] = pd.to_datetime(regs_df_2_clean['passage_date'])
regs_df_2_clean['effective_date'] = pd.to_datetime(regs_df_2_clean['effective_date'])


In [18]:
df = pd.concat([regs_df_1_clean, regs_df_2_clean], ignore_index=True)

assert df.duplicated().sum() == 0

df = df.sort_values(by=['state', 'city'], ascending=True, kind='stable')
df = df.reset_index(drop=True)

df.to_csv(OUTPUT_FILEPATH, index=False)
